# Run Broadcasting Experiments

Use this notebook for the main protocol workflow: exact simulation, QEC Monte Carlo sampling, a single IBM hardware point, or an IBM hardware tau sweep. Results are saved through the unified JSON schema in `results/`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from broadcasting import (
    ExactBackend,
    HardwareBackend,
    ProtocolConfig,
    SamplingBackend,
    load_run,
    save_run,
)
from broadcasting.plotting import plot_fidelity_vs_noise, plot_run_sweep, save_figure

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120})


## Configuration

Set `MODE` to `exact`, `sampling`, `hardware`, or `hardware_tau_sweep`.


In [ ]:
MODE = "exact"

M = 1
N = 2
alpha = 1.0 / np.sqrt(2)
use_qec = False
outcomes = [0] * M
seed = 42

rng = np.random.default_rng(seed)
nt = 1
theta_samples = rng.uniform(0, 2 * np.pi, size=(nt, M)).tolist()
thetas = theta_samples[0]

p_list = np.linspace(0, 1, 50).tolist()
n_samples = 1000

tau_values = np.linspace(0, 6000, 121).astype(int).tolist()
tau = tau_values[0]

IBM_CHANNEL = "ibm_quantum"
IBM_INSTANCE = "ibm-q/open/main"
IBM_BACKEND = "ibm_brisbane"
OPTIMIZATION_LEVEL = 3
SHOTS = 4096

SAVE_FIGURES = False
FIGURE_DIR = Path("figures")

print(f"Mode={MODE}  M={M}  N={N}  QEC={use_qec}")
print("theta samples:")
for i, sample in enumerate(theta_samples):
    print(f"  {i}: {np.array(sample)}")


## Run


In [ ]:
if MODE == "sampling" and not use_qec:
    raise ValueError("MODE='sampling' requires use_qec=True.")

config = ProtocolConfig(
    M=M,
    N=N,
    alpha=alpha,
    thetas=thetas,
    p_list=p_list if MODE in {"exact", "sampling"} else [],
    use_qec=use_qec,
    outcomes_list=outcomes,
    tau=tau if MODE == "hardware" else None,
    n_samples=n_samples if MODE == "sampling" else None,
    seed=seed,
)

if MODE == "exact":
    backend = ExactBackend()
    result = backend.run(config)
elif MODE == "sampling":
    backend = SamplingBackend(n_samples=n_samples, seed=seed)
    result = backend.run(config)
elif MODE in {"hardware", "hardware_tau_sweep"}:
    from qiskit_ibm_runtime import QiskitRuntimeService

    service = QiskitRuntimeService(channel=IBM_CHANNEL, instance=IBM_INSTANCE)
    backend = HardwareBackend(
        service=service,
        backend_name=IBM_BACKEND,
        shots=SHOTS,
        optimization_level=OPTIMIZATION_LEVEL,
    )
    if MODE == "hardware":
        result = backend.run(config)
    else:
        result = backend.run_tau_sweep(config, tau_values, theta_samples=theta_samples)
else:
    raise ValueError(f"Unknown MODE: {MODE}")

print(f"Recorded mode: {result.metadata.get('mode')}")
print(f"Fidelity array shape: {np.asarray(result.fidelities).shape}")


## Save And Plot


In [ ]:
out_path = save_run(result, config)
saved_run = load_run(out_path)
print(f"Saved to {out_path}")

if saved_run["sweep"]["axis"] == "p":
    fids = np.asarray(saved_run["fidelities"], dtype=float)
    fig = plot_fidelity_vs_noise(
        np.asarray(saved_run["sweep"]["values"], dtype=float),
        {f"Receiver {i}": fids[:, i] for i in range(saved_run["N"])},
        mode_label=saved_run.get("backend", MODE),
        protocol_info={"M": M, "N": N, "use_qec": use_qec},
        show=False,
    )
else:
    fig = plot_run_sweep(
        saved_run,
        tau_scale=4e-3,
        tau_label="Idle delay (us)",
        show=False,
    )

plt.tight_layout()
if SAVE_FIGURES:
    figure_path = FIGURE_DIR / f"{Path(out_path).stem}.png"
    save_figure(fig, figure_path)
    print(f"Saved title-free figure to {figure_path}")
plt.show()


## Optional Exact Vs Sampling Overlay

`SamplingBackend` is for the QEC path, so this comparison runs only when `use_qec=True`.


In [ ]:
RUN_COMPARISON = False

if RUN_COMPARISON and use_qec:
    compare_config = ProtocolConfig(
        M=M,
        N=N,
        alpha=alpha,
        thetas=thetas,
        p_list=p_list,
        use_qec=True,
        outcomes_list=outcomes,
        seed=seed,
    )
    exact = ExactBackend().run(compare_config)
    sampled = SamplingBackend(n_samples=5000, seed=seed).run(compare_config)

    p_arr = np.asarray(p_list, dtype=float)
    exact_avg = np.asarray(exact.fidelities, dtype=float).mean(axis=1)
    sampled_avg = np.asarray(sampled.fidelities, dtype=float).mean(axis=1)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(p_arr, exact_avg, label="Exact")
    ax.plot(p_arr, sampled_avg, "--", label="Sampling")
    ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
    ax.set_xlabel("Depolarizing probability p")
    ax.set_ylabel("Average fidelity")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "exact_vs_sampling.png")
    plt.show()
elif RUN_COMPARISON:
    print("Set use_qec=True before running the sampling comparison.")


## Optional Sampling Convergence


In [ ]:
RUN_CONVERGENCE = False

if RUN_CONVERGENCE:
    conv_config = ProtocolConfig(
        M=1,
        N=2,
        alpha=1.0 / np.sqrt(2),
        thetas=[0.0],
        p_list=np.linspace(0, 1, 21).tolist(),
        use_qec=True,
        outcomes_list=[0],
        seed=0,
    )
    exact_fids = np.asarray(ExactBackend().run(conv_config).fidelities)
    p_arr = np.asarray(conv_config.p_list)
    n_sweep = [50, 100, 200, 500, 1000, 2000, 5000, 10000]
    errors = []

    for ns in n_sweep:
        sampled = SamplingBackend(n_samples=ns, seed=0).run(conv_config)
        diff = np.abs(np.asarray(sampled.fidelities) - exact_fids)
        errors.append(np.trapz(diff, p_arr, axis=0).sum())
        print(f"n={ns:>5}: total area error={errors[-1]:.4f}")

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.semilogx(n_sweep, errors, "o-", label="Observed")
    ax.semilogx(
        n_sweep,
        errors[0] * np.sqrt(n_sweep[0]) / np.sqrt(n_sweep),
        ":",
        color="gray",
        label="1/sqrt(n)",
    )
    ax.set_xlabel("Number of samples")
    ax.set_ylabel("Error area")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend()
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "sampling_convergence.png")
    plt.show()
